# Demo 8 — Tree of Thoughts for Incident Root-Cause Analysis
## Session 6: Advanced Prompt Engineering · Optional / Extra

> **MLflow as the tape recorder.** Every prompt sent, every response returned, every hypothesis scored — captured automatically by `mlflow.openai.autolog()` and the `@mlflow.trace` decorators below. Open the MLflow UI to replay every input and output the model saw.

**ToT on a real-feeling production incident. Same scaffolding as D1, different evaluator. Slide 7's promise, applied.**

**Story arc:**
1. PagerDuty fires — payments-api p99 at 12s, 5xx at 18%.
2. Bundle the *evidence package*: log snippets + metrics summary + deploy history.
3. **Propose** k=4 candidate root-cause hypotheses.
4. **Evaluate** each hypothesis against the evidence (LLM-as-judge, 1–10).
5. **Search**: expand the top-`beam` survivors, ask sub-evidence questions, score again.
6. Aggregate the top branch into a final RCA + remediation recommendation.

**Why this notebook exists.** D1 demonstrates the ToT *mechanism* on a toy (Game-of-24) where a deterministic verifier exists. D8 shows the same scaffolding applied to the slide-7 promise — production RCA — where the value function is an LLM judge scoring against real evidence.

**Citation:** Yao et al. 2023, *Tree of Thoughts: Deliberate Problem Solving with Large Language Models* — [arXiv:2305.10601](https://arxiv.org/abs/2305.10601).

**Cost budget for a full run:** ~$0.05 on `gpt-4o` (4 root hypotheses × sub-evidence expansion + final aggregation ≈ 30 calls). $0 on local Ollama.

## Setup — env-var driven (mirrors D1)

Same env-var contract as every other Session-6 notebook. NO mock-mode fallback — fails loudly if Ollama is down.

```bash
export MLFLOW_TRACKING_URI=http://127.0.0.1:5000
export MLFLOW_EXPERIMENT=session6/demo_08_tot_rca
export OPENAI_BASE_URL=https://api.openai.com/v1   # or http://localhost:11434/v1 for Ollama
export OPENAI_API_KEY=sk-...                       # or 'ollama' for local
export MODEL=qwen2.5-coder:14b                     # RCA needs strong reasoning
```

In [ ]:
# -- Setup -----------------------------------------------------------------
import os
import json
import time
import re
from typing import Optional

import pandas as pd
import mlflow
from mlflow.entities import SpanType
from openai import OpenAI

# ─── MLflow tracking ───────────────────────────────────────────
MLFLOW_TRACKING_URI      = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
MLFLOW_TRACKING_USERNAME = os.getenv("MLFLOW_TRACKING_USERNAME")
MLFLOW_TRACKING_PASSWORD = os.getenv("MLFLOW_TRACKING_PASSWORD")
MLFLOW_REGISTRY_URI      = os.getenv("MLFLOW_REGISTRY_URI", MLFLOW_TRACKING_URI)
MLFLOW_EXPERIMENT        = os.getenv("MLFLOW_EXPERIMENT", "session6/demo_08_tot_rca")

if MLFLOW_TRACKING_USERNAME:
    os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_TRACKING_USERNAME
if MLFLOW_TRACKING_PASSWORD:
    os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_TRACKING_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_registry_uri(MLFLOW_REGISTRY_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

# ─── LLM (OpenAI-compatible: OpenAI, Azure, vLLM, Ollama, Together, etc.) ────
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "http://localhost:11434/v1")  # Ollama default
OPENAI_API_KEY  = os.getenv("OPENAI_API_KEY", "ollama")
MODEL           = os.getenv("MODEL", "qwen2.5-coder:14b")  # RCA needs strong reasoning

# No mock-mode fallback. If Ollama isn't running, fail loud.
if not OPENAI_API_KEY:
    raise SystemExit("Set OPENAI_API_KEY or start Ollama (defaults to api_key='ollama').")

TEMPERATURE = float(os.getenv("TEMPERATURE", "0.0"))

# Per-1K-token pricing (USD). Update at each model release.
PRICING = {
    "gpt-4o":             {"in": 0.0025,   "out": 0.010},
    "gpt-4o-mini":        {"in": 0.00015,  "out": 0.0006},
    "o1-mini":            {"in": 0.003,    "out": 0.012},
    "claude-sonnet-4-6":  {"in": 0.003,    "out": 0.015},
    # Hypothetical local-LLM pricing (electricity + amortised GPU per 1K tokens)
    "qwen2.5-coder:14b":  {"in": 0.00005,  "out": 0.00005},
    "qwen2.5-coder:7b":   {"in": 0.00002,  "out": 0.00002},
    "llama3.1:8b":        {"in": 0.00002,  "out": 0.00002},
}

def price_of(model: str, usage) -> float:
    if usage is None:
        return 0.0
    p = PRICING.get(model, {"in": 0.00005, "out": 0.00005})
    return usage.prompt_tokens / 1000 * p["in"] + usage.completion_tokens / 1000 * p["out"]


def tag_cost_latency(latency_ms: float, cost_usd: float) -> None:
    span = mlflow.get_current_active_span()
    if span:
        span.set_attribute("latency_ms", latency_ms)
        span.set_attribute("cost_usd", cost_usd)
    try:
        mlflow.update_current_trace(tags={
            "cost_usd":   f"{cost_usd:.6f}",
            "latency_ms": f"{latency_ms:.0f}",
        })
    except Exception:
        pass


client = OpenAI(base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY)
mlflow.openai.autolog()

print(f"MLflow:  {MLFLOW_TRACKING_URI}  (exp: {MLFLOW_EXPERIMENT})")
print(f"LLM:     {OPENAI_BASE_URL}  model={MODEL}  temperature={TEMPERATURE}")
print(f"UI:      open {MLFLOW_TRACKING_URI}  → Experiments → {MLFLOW_EXPERIMENT}")

## The incident

**PagerDuty fires at 14:02:17 UTC.**

- `payments-api` p99 latency: **12s** (normal: ~300ms)
- 5xx error rate: **18%** (normal: <0.1%)
- Customer impact: checkout flow degraded

**Your job:** find the root cause and propose remediation.

**Why ToT here?** RCA is the *exact* pattern ToT was designed for — multiple candidate hypotheses, each scorable against evidence, and *catastrophic early-commit* (Self-Consistency, slide 2, voted unanimously for the wrong answer).

In [ ]:
# -- Incident evidence package ---------------------------------------------
# Realistic Loki output: ISO timestamps, service tags, mixed signal + noise.

LOG_SNIPPETS = """\
2025-05-22T14:01:48Z service=payments-api severity=INFO  msg="request ok" path=/v1/charge dur_ms=287
2025-05-22T14:02:01Z service=payments-api severity=INFO  msg="request ok" path=/v1/charge dur_ms=312
2025-05-22T14:02:14Z service=payments-api severity=INFO  msg="GC pause" duration_ms=42 heap_used_mb=842
2025-05-22T14:02:17Z service=payments-api severity=ERROR msg="HikariPool-1 - Connection is not available, request timed out after 30000ms" pool_active=50 pool_max=50 pool_waiting=128
2025-05-22T14:02:18Z service=payments-api severity=ERROR msg="HikariPool-1 - Connection is not available, request timed out after 30000ms" pool_active=50 pool_max=50 pool_waiting=141
2025-05-22T14:02:19Z service=payments-api severity=ERROR msg="java.sql.SQLTransientConnectionException" stack="HikariPool$PoolEntryFactory ... acquireConnection"
2025-05-22T14:02:21Z service=payments-api severity=WARN  msg="slow query detected" query="SELECT * FROM ledger WHERE customer_id=?" dur_ms=11842
2025-05-22T14:02:22Z service=payments-api severity=ERROR msg="HikariPool-1 - Connection is not available, request timed out after 30000ms" pool_active=50 pool_max=50 pool_waiting=156
2025-05-22T14:02:24Z service=payments-api severity=INFO  msg="GC pause" duration_ms=38 heap_used_mb=851
2025-05-22T14:02:27Z service=payments-api severity=ERROR msg="downstream auth-service responded 502" upstream_dur_ms=29994
2025-05-22T14:02:31Z service=payments-api severity=ERROR msg="HikariPool-1 - Connection is not available, request timed out after 30000ms" pool_active=50 pool_max=50 pool_waiting=178
2025-05-22T14:02:33Z service=payments-api severity=WARN  msg="slow query detected" query="UPDATE ledger SET balance=? WHERE id=?" dur_ms=14217
2025-05-22T14:02:36Z service=payments-api severity=ERROR msg="java.sql.SQLTransientConnectionException" stack="HikariPool$PoolEntryFactory ... acquireConnection"
2025-05-22T14:02:38Z service=payments-api severity=INFO  msg="healthcheck ok" dur_ms=4
2025-05-22T14:02:41Z service=payments-api severity=ERROR msg="HikariPool-1 - Connection is not available, request timed out after 30000ms" pool_active=50 pool_max=50 pool_waiting=192
2025-05-22T14:02:44Z service=payments-api severity=WARN  msg="DB connection acquire latency" acquire_ms=29812 pool_waiting=204
2025-05-22T14:02:48Z service=payments-api severity=ERROR msg="HikariPool-1 - Connection is not available, request timed out after 30000ms" pool_active=50 pool_max=50 pool_waiting=221
"""

METRICS_SUMMARY = {
    "window":          "14:00–14:30 UTC, 2025-05-22",
    "p50_ms_before":   148,
    "p50_ms_after":    8420,
    "p99_ms_before":   312,
    "p99_ms_after":    12017,
    "error_rate_before":  0.0006,
    "error_rate_after":   0.184,
    "db_pool_acquire_ms_before":  3.2,
    "db_pool_acquire_ms_after":   29812,
    "db_pool_active":  50,
    "db_pool_max":     50,
    "cpu_pct_after":   34,
    "heap_used_pct_after": 41,
    "qps_before":      1180,
    "qps_after":       1244,
}

DEPLOY_HISTORY = [
    {"ts": "2025-05-22T13:53:11Z", "service": "payments-api",  "sha": "a91f4c2", "author": "alice", "note": "refactor: extract LedgerRepository"},
    {"ts": "2025-05-22T11:18:02Z", "service": "auth-service",  "sha": "77ce119", "author": "bob",   "note": "chore: bump deps"},
    {"ts": "2025-05-21T22:04:33Z", "service": "payments-api",  "sha": "3a01dd0", "author": "carol", "note": "feat: new ledger query in checkout path (no index on customer_id yet)"},
    {"ts": "2025-05-21T16:40:08Z", "service": "ledger-db",     "sha": "",        "author": "sre",   "note": "infra: scale read replicas 2→3"},
    {"ts": "2025-05-20T09:12:55Z", "service": "payments-api",  "sha": "e7b89a4", "author": "dan",   "note": "chore: update logging library"},
]

EVIDENCE = {
    "incident":        "payments-api p99 12s, 5xx 18%, fired 14:02:17 UTC",
    "logs":            LOG_SNIPPETS,
    "metrics":         METRICS_SUMMARY,
    "deploy_history":  DEPLOY_HISTORY,
}

print(f"Evidence package: {len(LOG_SNIPPETS.splitlines())} log lines, {len(METRICS_SUMMARY)} metric fields, {len(DEPLOY_HISTORY)} deploys.")
print(f"Spike onset: 14:02:17Z. Most recent deploy to payments-api: 13:53:11Z (9 min earlier — the red-herring hypothesis).")

## ToT setup

Propose **k=4** candidate root-cause hypotheses. Evaluate each against the evidence (LLM-as-judge, 1–10). Keep top-**beam=2**. Expand those by asking sub-evidence questions ("would X be observable if this hypothesis is correct?"). Aggregate the highest-scoring branch into a final RCA + remediation.

**Why this differs from D1:** the evaluator is no longer deterministic. It's an LLM-as-judge scoring hypothesis-vs-evidence on a 1–10 scale. Per slide-25 / Misconception 0 / Huang 2310.01798 — the evaluator is a *separate call* with `temperature=0` and a different prompt than the proposer, so it's not the same model critiquing its own draft. That's the minimum-viable external-ish verifier when no deterministic ground truth exists.

In [ ]:
# -- Propose root hypotheses ------------------------------------------------

PROPOSE_PROMPT = """\
You are an SRE doing root-cause analysis. An incident just fired.

INCIDENT: {incident}

EVIDENCE (logs):
{logs}

EVIDENCE (metrics): {metrics}

EVIDENCE (recent deploys): {deploys}

Propose exactly {k} DISTINCT candidate root-cause hypotheses. Each one must be
a different category (don't list two flavours of the same cause). Include at least
one hypothesis that engineers would jump to from surface correlation, even if you
think it's wrong.

Output a JSON array of objects with keys "name" (short) and "rationale" (1-2 sentences).
Output ONLY the JSON array, no prose, no markdown fences.
"""

PROPOSE_TEMPERATURE = max(0.7, TEMPERATURE if TEMPERATURE >= 0.5 else 0.7)


def _strip_fences(text: str) -> str:
    t = text.strip()
    if t.startswith("```"):
        t = re.sub(r"^```[a-zA-Z]*\n?", "", t)
        t = re.sub(r"\n?```\s*$", "", t)
    return t.strip()


@mlflow.trace(name="propose", span_type=SpanType.LLM, attributes={"step": "hypothesis_generation"})
def propose_hypotheses(evidence: dict, k: int = 4) -> list[dict]:
    prompt = PROPOSE_PROMPT.format(
        incident=evidence["incident"],
        logs=evidence["logs"],
        metrics=json.dumps(evidence["metrics"]),
        deploys=json.dumps(evidence["deploy_history"]),
        k=k,
    )
    span = mlflow.get_current_active_span()
    if span:
        span.set_inputs({"prompt": prompt, "params": {"model": MODEL, "k": k, "temperature": PROPOSE_TEMPERATURE}})
    t0 = time.time()
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=PROPOSE_TEMPERATURE,
        max_tokens=900,
    )
    latency_ms = (time.time() - t0) * 1000
    cost = price_of(MODEL, resp.usage)
    tag_cost_latency(latency_ms, cost)
    text = resp.choices[0].message.content or "[]"
    try:
        hyps = json.loads(_strip_fences(text))
    except json.JSONDecodeError:
        # Fallback: find first JSON array in the response.
        m = re.search(r"\[.*\]", text, re.DOTALL)
        hyps = json.loads(m.group(0)) if m else []
    hyps = [h for h in hyps if isinstance(h, dict) and "name" in h][:k]
    if span:
        span.set_outputs({"text": text, "hypotheses": hyps})
    return hyps


print("propose_hypotheses ready.")

In [ ]:
# -- Evaluate one hypothesis against the evidence --------------------------

EVAL_PROMPT = """\
You are scoring a root-cause hypothesis against incident evidence.

INCIDENT: {incident}
LOGS:
{logs}
METRICS: {metrics}
RECENT DEPLOYS: {deploys}

HYPOTHESIS: {name}
RATIONALE:  {rationale}

Score 1–10 how well this hypothesis is *supported by the evidence chain*, NOT by surface correlation.
- 10 = direct log + metric evidence at the spike timestamp
- 7  = consistent with metrics but indirect log support
- 4  = plausible category but no direct evidence in this window
- 1  = contradicted by evidence (e.g., wrong timestamp, wrong service)

Output ONLY a single integer 1–10. No prose, no explanation.
"""


@mlflow.trace(name="evaluate", span_type=SpanType.LLM, attributes={"step": "hypothesis_scoring"})
def evaluate_hypothesis(hypothesis: dict, evidence: dict) -> int:
    prompt = EVAL_PROMPT.format(
        incident=evidence["incident"],
        logs=evidence["logs"],
        metrics=json.dumps(evidence["metrics"]),
        deploys=json.dumps(evidence["deploy_history"]),
        name=hypothesis.get("name", ""),
        rationale=hypothesis.get("rationale", ""),
    )
    span = mlflow.get_current_active_span()
    if span:
        span.set_inputs({"prompt": prompt, "params": {"model": MODEL, "temperature": 0.0, "hypothesis": hypothesis.get("name")}})
    t0 = time.time()
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,   # judge MUST be deterministic
        max_tokens=8,
    )
    latency_ms = (time.time() - t0) * 1000
    cost = price_of(MODEL, resp.usage)
    tag_cost_latency(latency_ms, cost)
    text = (resp.choices[0].message.content or "").strip()
    m = re.search(r"\b(10|[1-9])\b", text)
    score = int(m.group(1)) if m else 1
    score = max(1, min(10, score))
    if span:
        span.set_outputs({"text": text, "score": score})
        span.set_attribute("score", score)
    return score


SUBEVIDENCE_PROMPT = """\
You are testing a root-cause hypothesis by asking what sub-evidence WOULD be observable if it were true.

HYPOTHESIS: {name}
RATIONALE:  {rationale}

Original evidence:
LOGS: {logs}
METRICS: {metrics}

Propose exactly {k} sub-evidence checks. Each one is a question of the form
"would <specific observation> be present if <hypothesis>?" then state whether the
supplied evidence answers YES, NO, or AMBIGUOUS.

Output JSON array of objects with keys "check" and "answer" (one of YES/NO/AMBIGUOUS).
Output ONLY the JSON array.
"""


@mlflow.trace(name="sub_evidence", span_type=SpanType.LLM, attributes={"step": "sub_evidence_check"})
def sub_evidence(hypothesis: dict, evidence: dict, k: int = 3) -> list[dict]:
    prompt = SUBEVIDENCE_PROMPT.format(
        name=hypothesis.get("name", ""),
        rationale=hypothesis.get("rationale", ""),
        logs=evidence["logs"],
        metrics=json.dumps(evidence["metrics"]),
        k=k,
    )
    span = mlflow.get_current_active_span()
    if span:
        span.set_inputs({"prompt": prompt, "params": {"model": MODEL, "k": k, "temperature": 0.0}})
    t0 = time.time()
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=400,
    )
    latency_ms = (time.time() - t0) * 1000
    cost = price_of(MODEL, resp.usage)
    tag_cost_latency(latency_ms, cost)
    text = resp.choices[0].message.content or "[]"
    try:
        checks = json.loads(_strip_fences(text))
    except json.JSONDecodeError:
        m = re.search(r"\[.*\]", text, re.DOTALL)
        checks = json.loads(m.group(0)) if m else []
    checks = [c for c in checks if isinstance(c, dict)][:k]
    if span:
        span.set_outputs({"text": text, "checks": checks})
    return checks


print("evaluate_hypothesis + sub_evidence ready.")

In [ ]:
# -- Final aggregation: top branch -> RCA + remediation -------------------

AGGREGATE_PROMPT = """\
You are writing the final root-cause analysis for an incident review.

INCIDENT: {incident}

TOP HYPOTHESIS (selected by ToT search, score {score}/10):
Name:      {name}
Rationale: {rationale}

Sub-evidence checks that survived scoring:
{checks}

Original evidence: {evidence_summary}

Write a tight RCA. Output exactly these 4 sections, each 1-3 sentences:
1. ROOT CAUSE
2. CHAIN OF EVENTS
3. IMMEDIATE REMEDIATION (next 30 min)
4. LONG-TERM FIX (next sprint)

Be specific. No filler.
"""


@mlflow.trace(name="aggregate", span_type=SpanType.LLM, attributes={"step": "final_rca"})
def aggregate_rca(top: dict, evidence: dict) -> str:
    prompt = AGGREGATE_PROMPT.format(
        incident=evidence["incident"],
        score=top["score"],
        name=top["name"],
        rationale=top["rationale"],
        checks=json.dumps(top.get("checks", []), indent=2),
        evidence_summary=f"{len(evidence['logs'].splitlines())} log lines, p99 {evidence['metrics']['p99_ms_after']}ms, pool_waiting peaked at 221, most recent deploy was 9 min before spike",
    )
    span = mlflow.get_current_active_span()
    if span:
        span.set_inputs({"prompt": prompt, "params": {"model": MODEL, "temperature": 0.0}})
    t0 = time.time()
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=600,
    )
    latency_ms = (time.time() - t0) * 1000
    cost = price_of(MODEL, resp.usage)
    tag_cost_latency(latency_ms, cost)
    text = resp.choices[0].message.content or ""
    if span:
        span.set_outputs({"text": text})
    return text


print("aggregate_rca ready.")

In [ ]:
# -- tot_rca: BFS over the hypothesis tree, fully traced -------------------

@mlflow.trace(
    name="tot_rca",
    span_type=SpanType.CHAIN,
    attributes={"beam": 2, "depth": 3, "k": 4},
)
def tot_rca(evidence: dict, beam: int = 2, depth: int = 3, k: int = 4) -> dict:
    """BFS Tree-of-Thoughts over root-cause hypotheses. Returns the best survivor + final RCA."""
    root_span = mlflow.get_current_active_span()
    if root_span:
        root_span.set_inputs({"prompt": f"RCA for: {evidence['incident']}",
                              "params": {"model": MODEL, "beam": beam, "depth": depth, "k": k}})

    all_nodes: list[dict] = []

    # ---- Round 0: propose k root hypotheses, score each, keep top-beam ----
    with mlflow.start_span(name="expand_root", span_type=SpanType.CHAIN) as sp:
        sp.set_inputs({"prompt": "propose k root hypotheses", "params": {"k": k}})
        hyps = propose_hypotheses(evidence, k=k)
        scored_roots = []
        for i, h in enumerate(hyps):
            score = evaluate_hypothesis(h, evidence)
            node = {
                "id":        f"r.{i}",
                "depth":     0,
                "name":      h.get("name", ""),
                "rationale": h.get("rationale", ""),
                "score":     score,
                "parent":    "r",
                "pruned":    False,
                "checks":    [],
            }
            scored_roots.append(node)
            all_nodes.append(node)
            mlflow.log_metric(f"node_score.{node['id']}", score, step=0)
        sp.set_outputs({"text": f"{len(scored_roots)} hypotheses scored", "hypotheses": scored_roots})
        sp.set_attribute("n_children", len(scored_roots))

    # Prune to top-beam.
    kept = sorted(scored_roots, key=lambda n: -n["score"])[:beam]
    kept_ids = {n["id"] for n in kept}
    for n in scored_roots:
        if n["id"] not in kept_ids:
            n["pruned"] = True
    mlflow.log_metric("frontier_size", len(kept), step=0)
    mlflow.log_metric("pruned_count", len(scored_roots) - len(kept), step=0)

    # ---- Round 1: for each surviving hypothesis, ask sub-evidence + rescore ----
    for survivor in kept:
        with mlflow.start_span(name=f"expand_{survivor['id']}", span_type=SpanType.CHAIN) as sp:
            sp.set_inputs({"prompt": f"sub-evidence for {survivor['name']}",
                           "params": {"node_id": survivor["id"]}})
            checks = sub_evidence({"name": survivor["name"], "rationale": survivor["rationale"]}, evidence, k=3)
            survivor["checks"] = checks
            # Rescore in light of sub-evidence answers.
            yes_count = sum(1 for c in checks if str(c.get("answer", "")).upper() == "YES")
            adj = min(10, survivor["score"] + yes_count) if checks else survivor["score"]
            survivor["score"] = adj
            mlflow.log_metric(f"node_score.{survivor['id']}", adj, step=1)
            sp.set_outputs({"text": f"{len(checks)} checks, {yes_count} YES, adjusted={adj}",
                            "checks": checks, "adjusted_score": adj})
            sp.set_attribute("score", adj)
            sp.set_attribute("sub_yes_count", yes_count)

    # ---- Round 2: pick best, aggregate to final RCA ----
    best = max(kept, key=lambda n: n["score"])
    final_rca = aggregate_rca(best, evidence)

    # ---- Log the full tree as an artifact ----
    tree_df = pd.DataFrame([
        {
            "id":        n["id"],
            "parent":    n["parent"],
            "depth":     n["depth"],
            "name":      n["name"],
            "score":     n["score"],
            "pruned":    n["pruned"],
            "rationale": n["rationale"],
            "n_checks":  len(n["checks"]),
        }
        for n in all_nodes
    ])
    mlflow.log_table(data=tree_df.to_dict(orient="list"), artifact_file="rca_tree.json")

    mlflow.update_current_trace(tags={
        "final_root_cause": best["name"][:80],
        "best_score":       f"{best['score']}",
        "tot_best_score":   f"{best['score']}",
        "nodes_total":      str(len(all_nodes)),
        "incident":         evidence["incident"][:80],
    })
    result = {
        "best":   best,
        "all_nodes": all_nodes,
        "rca":    final_rca,
    }
    if root_span:
        root_span.set_outputs({"text": f"best={best['name']} score={best['score']}",
                               "rca":  final_rca,
                               "n_nodes": len(all_nodes)})
    return result


print("tot_rca ready.")

In [ ]:
# -- Run the RCA hypothesis tree on the incident --------------------------

result = tot_rca(EVIDENCE, beam=2, depth=3, k=4)

print("=" * 72)
print("FULL HYPOTHESIS TREE")
print("=" * 72)
for n in result["all_nodes"]:
    mark = "PRUNED" if n["pruned"] else "KEPT  "
    print(f"  [{mark}] {n['id']}  score={n['score']:>2}  {n['name']}")
    if n["checks"]:
        for c in n["checks"]:
            print(f"            └─ {c.get('answer','?'):>9}  {c.get('check','')[:90]}")

print()
print("=" * 72)
print(f"TOP HYPOTHESIS: {result['best']['name']}  (score {result['best']['score']}/10)")
print("=" * 72)
print(result["rca"])

## The point

The model SHOULD land on **DB connection pool exhaustion** as the top hypothesis. The evidence is unambiguous:

- `pool_active=50 pool_max=50 pool_waiting` climbing from 128 → 221 during the spike
- `db_pool_acquire_ms_after = 29812` (~30s, the configured timeout)
- Slow queries on `ledger` showing acquire-and-hold pattern

It SHOULD prune **deploy rollback** — the surface-correlation hypothesis (slide 2 ground truth). The most recent deploy was 9 minutes earlier, *before* the spike, and was an unrelated refactor. The evaluator scores this low because asking "would the 14:02 spike correlate with the 13:53 deploy?" returns AMBIGUOUS at best, and the log evidence points squarely at the connection pool.

**How the value function helped.** Self-Consistency (slide 2) sampled 5 chains and voted unanimously for deploy rollback — all 5 chains absorbed the same surface correlation between "5xx" and "recent deploy". ToT externalised the scoring step: each hypothesis is scored *against the evidence chain*, not against the prior. The deploy hypothesis loses on the timestamp-mismatch evidence.

**The deeper hypothesis tree could go further:** a likely *contributing cause* is the **new ledger query without an index** (deploy 3a01dd0 at 22:04 the night before — visible in `DEPLOY_HISTORY`). That deploy isn't the *trigger* (it shipped 16 hours earlier and worked fine through traffic peak), but combined with daily-batch traffic at 14:02, the unindexed query started holding pool connections long enough to exhaust them. A good run on `qwen2.5-coder:14b` will surface this in the LONG-TERM FIX section.

In [ ]:
# -- Query past high-scoring RCA runs (this trace becomes reproducible/queryable) --

past = mlflow.search_traces(
    experiment_names=[MLFLOW_EXPERIMENT],
    filter_string="tags.tot_best_score > '7'",
    max_results=5,
)

if len(past) == 0:
    print("No high-scoring RCA traces yet — run the tot_rca cell a few times.")
else:
    cols = [c for c in ["trace_id", "tags.incident", "tags.final_root_cause",
                         "tags.tot_best_score", "tags.nodes_total"]
            if c in past.columns]
    print(past[cols].to_string(index=False))

## 🎥 Replay every input + output in MLflow

Every prompt and every response from this demo is now in MLflow. Open the UI and click any trace to see the full I/O log + the `rca_tree.json` artifact (sortable table of every hypothesis, score, and pruning decision).

In [ ]:
import IPython.display as D

ui = MLFLOW_TRACKING_URI.rstrip("/")
exp = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)
exp_id = exp.experiment_id if exp else "0"
link = f"{ui}/#/experiments/{exp_id}?searchFilter=&orderByKey=attributes.start_time&orderByAsc=false"
D.display(D.Markdown(f"**Open the experiment in MLflow:** [{link}]({link})"))

recent = mlflow.search_traces(experiment_ids=[exp_id], max_results=5)
recent[["request_id", "timestamp_ms", "execution_time_ms", "tags"]] if not recent.empty else "No traces yet — run the cells above."

## ⚠️ Reasoning model caveat (same as D6/D7)

This notebook assumes a **non-reasoning** base model. On `o1` / `o3` / Claude Extended Thinking / Gemini Thinking:
- Drop the ToT scaffolding entirely — feed the evidence package directly. The model does the propose/evaluate/prune internally during its thinking phase.
- Use `developer` role instead of `system` on `o1-2024-12-17+`.
- Never pass OpenAI `reasoning_effort` and Gemini `thinking_level` through the same OpenAI-compatible adapter.
- See Block 5 of the slides for the full "what dies / what survives" list.

**The ToT scaffolding is what you ship on a non-reasoning base model when production cost rules out o3.** On a reasoning model the slide-7 RCA tree collapses to a single prompt — paying twice for the search.